<a href="https://colab.research.google.com/github/emartinez6255/PythonDroneCode/blob/training-setup/train_yolo_smoke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Ultralytics
!pip install ultralytics

import ultralytics
from ultralytics import YOLO
import os

# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU available: True


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Clone Github Repository
!git clone https://github.com/emartinez6255/PythonDroneCode.git
%cd PythonDroneCode
!git fetch origin
!git checkout training-setup

Cloning into 'PythonDroneCode'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 45 (delta 8), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 5.83 MiB | 11.20 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/PythonDroneCode
Branch 'training-setup' set up to track remote branch 'training-setup' from 'origin'.
Switched to a new branch 'training-setup'


In [4]:
# Setup Project Directory

# Move into my project folder on Drive
%cd /content/PythonDroneCode

# Check that files are there
!ls

/content/PythonDroneCode
Drone_Environment.py  requirements.txt	src


In [ ]:
!pip install -r requirements.txt

In [5]:
%cd /content
%mkdir data
%ls

/content
data/  drive/  PythonDroneCode/  sample_data/


In [6]:
%%writefile /content/data_config.yaml
# Dataset root directory (absolute path in Colab)
path: /content/PythonDroneCode/data

# Relative paths from the 'path' directory above
train: content/data/train
val: content/data/val
test: content/data/test

# Classes
nc: 1
names: ['human']

Writing /content/data_config.yaml


In [7]:
# Importing dataset from KaggleHub
# SARD - Search and Rescue Dataset

# Install dependencies
!pip install opendatasets
!pip install pandas

import opendatasets as od
import pandas

od.download(
    "https://www.kaggle.com/datasets/nikolasgegenava/sard-search-and-rescue/code/data"
)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: emuno086
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/nikolasgegenava/sard-search-and-rescue


100%|██████████| 626M/626M [00:17<00:00, 38.0MB/s]


In [8]:
import os
import shutil

# Where the data currently is, and where we want it to go
source_base = '/content/sard-search-and-rescue/search-and-rescue/'
dest_base = '/content/data/'

# Kaggle datasets often use 'valid' instead of 'val'
split_mapping = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}

print("Starting dataset reorganization...\n")

for source_split, dest_split in split_mapping.items():
    source_split_dir = os.path.join(source_base, source_split)

    if os.path.exists(source_split_dir):
        print(f"Processing '{source_split}' data...")

        # 1. Create target images and labels directories
        os.makedirs(os.path.join(dest_base, dest_split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(dest_base, dest_split, 'labels'), exist_ok=True)

        # 2. Check if the original dataset already separated images and labels
        if os.path.exists(os.path.join(source_split_dir, 'images')):
            # Move contents of images and labels folders
            for item in os.listdir(os.path.join(source_split_dir, 'images')):
                shutil.move(os.path.join(source_split_dir, 'images', item), os.path.join(dest_base, dest_split, 'images', item))
            for item in os.listdir(os.path.join(source_split_dir, 'labels')):
                shutil.move(os.path.join(source_split_dir, 'labels', item), os.path.join(dest_base, dest_split, 'labels', item))
        else:
            # 3. If images and labels are mixed in the same folder, sort them by extension
            for item in os.listdir(source_split_dir):
                if item.endswith(('.jpg', '.jpeg', '.png')):
                    shutil.move(os.path.join(source_split_dir, item), os.path.join(dest_base, dest_split, 'images', item))
                elif item.endswith('.txt'):
                    shutil.move(os.path.join(source_split_dir, item), os.path.join(dest_base, dest_split, 'labels', item))

        print(f"Successfully moved {source_split} -> {dest_split}/")

print("\nAll done! Your dataset is now in the YOLO gold-standard format.")

Starting dataset reorganization...

Processing 'train' data...
Successfully moved train -> train/
Processing 'valid' data...
Successfully moved valid -> val/
Processing 'test' data...
Successfully moved test -> test/

All done! Your dataset is now in the YOLO gold-standard format.


In [10]:
# Download the pretrained yolov8n with base weights before transfer learning for our model
!yolo task=detect mode=train model=yolov8n.pt data=/content/data.yaml epochs=100 imgsz=640 batch=32 device=0 plots=True

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=T

In [9]:
import yaml
import os

# Define the YAML data for JUST the SARD dataset
data = {
    'train': '/content/data/train/',
    'val': '/content/data/val/',
    'test': '/content/data/test/',
    'nc': 1,  # Changed to 1 since we are only doing humans right now
    'names': ['human']
}

# Write to the file
file_path = '/content/data.yaml'
with open(file_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

# Verify it was created
if os.path.exists(file_path):
    print(f"Success! data.yaml exists at {file_path}")
    with open(file_path, 'r') as f:
        print("\nFile contents:")
        print(f.read())
else:
    print("Error: File still not created.")

Success! data.yaml exists at /content/data.yaml

File contents:
train: /content/data/train/
val: /content/data/val/
test: /content/data/test/
nc: 1
names:
- human



In [12]:
!yolo task=detect mode=predict model=/content/runs/detect/train/weights/best.pt source=/content/data/test/images/ conf=0.5 save=True

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

image 1/570 /content/data/test/images/gss1006_jpg.rf.32aa7f99ebd9eed1e6777ff5a3b64517.jpg: 640x640 1 human, 8.1ms
image 2/570 /content/data/test/images/gss1006_jpg.rf.7420665da812a124c7a8cfd6f9d67bbe.jpg: 640x640 (no detections), 6.5ms
image 3/570 /content/data/test/images/gss1006_jpg.rf.8ba83f6f0ac292da5003d3bcddd63f7a.jpg: 640x640 (no detections), 9.6ms
image 4/570 /content/data/test/images/gss1015_jpg.rf.0ffe8deefec932ca39ed0ce141726e51.jpg: 640x640 2 humans, 7.5ms
image 5/570 /content/data/test/images/gss1015_jpg.rf.5ace33228d5369ea6bf22118f6a5dca6.jpg: 640x640 2 humans, 6.9ms
image 6/570 /content/data/test/images/gss1015_jpg.rf.8a1644be346d5efbd5a9748bcaef8bb5.jpg: 640x640 2 humans, 6.8ms
image 7/570 /content/data/test/images/gss1015_jpg.rf.9ada927b3321306afa1ade7709bb6ad8.jpg: 640x640 1 human, 6.9ms
image 8/570 /conten

In [13]:
from google.colab import files
files.download('/content/runs/detect/train/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>